This notebook create a DuckDB connection with the Musicbrainz postgres running locally and pulls tables into parquet files

In [1]:
import os
import duckdb
from dotenv import load_dotenv

load_dotenv()

pg_host = os.getenv('PG_HOST', 'localhost')
pg_port = os.getenv('PG_PORT', '5432')
pg_dbname = os.getenv('PG_DBNAME', 'musicbrainz_db')
pg_user = os.getenv('PG_USER')
pg_password = os.getenv('PG_PASSWORD')

duck_con = duckdb.connect("musicbrainz.duckdb")

duck_con.execute("""
INSTALL postgres;
LOAD postgres;
""")

duck_con.execute(f"""
ATTACH IF NOT EXISTS 'host={pg_host} port={pg_port} dbname={pg_dbname} user={pg_user} password={pg_password}'
AS mb_pg
(TYPE postgres, READ_ONLY);
""")

In [ ]:
#import artists
duck_con.execute(f"""
COPY (
    SELECT
        id,
        name,
        begin_date_year AS artist_year,
        type,
        area,
        gender
    FROM mb_pg.musicbrainz.artist
    ORDER BY id
)
TO '../data/mb_artist.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

In [ ]:
#import artist tags
duck_con.execute(f"""
COPY (
    SELECT
        artist AS artist_id,
        tag AS tag_id,
        count AS tag_count
    FROM mb_pg.musicbrainz.artist_tag
    ORDER BY artist
)
TO '../data/mb_artist_tag.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

In [ ]:
#import artist ratings
duck_con.execute(f"""
COPY (
    SELECT
        id AS artist_id,
        rating,
        rating_count
    FROM mb_pg.musicbrainz.artist_meta
    WHERE rating IS NOT NULL
    ORDER BY rating_count DESC
)
TO '../data/mb_artist_ratings.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

In [ ]:
#import albums
duck_con.execute(f"""
COPY (
    SELECT
        id,
        name,
        artist_credit
    FROM mb_pg.musicbrainz.release_group
    WHERE type = 1
    ORDER BY id
)
TO '../data/mb_album.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

In [ ]:
#import album tags
duck_con.execute(f"""
COPY (
    SELECT
        t.release_group AS album_id,
        t.tag AS tag_id,
        t.count AS tag_count
    FROM mb_pg.musicbrainz.release_group_tag t
    JOIN mb_pg.musicbrainz.release_group rg ON t.release_group = rg.id
    WHERE rg.type = 1
      AND t.count > 0
    ORDER BY t.release_group
)
TO '../data/mb_album_tag.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

In [ ]:
#import album ratings
duck_con.execute(f"""
COPY (
    SELECT
        m.id AS album_id,
        m.rating,
        m.rating_count
    FROM mb_pg.musicbrainz.release_group_meta m
    JOIN mb_pg.musicbrainz.release_group rg ON m.id = rg.id
    WHERE rg.type = 1 
      AND m.rating IS NOT NULL
    ORDER BY m.rating_count DESC
)
TO '../data/mb_album_ratings.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

In [ ]:
#import album country
duck_con.execute(f"""
COPY (
    SELECT DISTINCT ON (r.release_group)
        r.release_group AS album_id,
        r.language,
        rc.country,
        rc.date_year AS album_year
    FROM mb_pg.musicbrainz.release r
    JOIN mb_pg.musicbrainz.release_country rc ON r.id = rc.release
    JOIN mb_pg.musicbrainz.release_group rg ON r.release_group = rg.id
    WHERE rg.type = 1
    ORDER BY r.release_group,
             rc.date_year ASC NULLS LAST,
             rc.date_month ASC NULLS LAST,
             rc.date_day ASC NULLS LAST
)
TO '../data/mb_album_country.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

In [ ]:
#import album label
duck_con.execute(f"""
COPY (
    SELECT DISTINCT ON (r.release_group)
        r.release_group AS album_id,
        rl.label AS label_id,
        l.type AS label_type,
        lt.tag AS tag_id,
        lt.count AS tag_count
    FROM mb_pg.musicbrainz.release_label rl
    JOIN mb_pg.musicbrainz.release r ON rl.release = r.id
    JOIN mb_pg.musicbrainz.release_group rg ON r.release_group = rg.id
    JOIN mb_pg.musicbrainz.label l ON rl.label = l.id
    JOIN mb_pg.musicbrainz.label_tag lt ON rl.label = lt.label
    WHERE rg.type = 1
    ORDER BY r.release_group,
             lt.count DESC NULLS LAST
)
TO '../data/mb_album_label.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

In [ ]:
#import artist credit
duck_con.execute(f"""
COPY (
    SELECT
        ac.id AS artist_credit,
        ac.name,
        ac.artist_count,
        ac.ref_count,
        acn.position,
        acn.artist AS artist_id,
        acn.name AS artist_name,
        acn.join_phrase
    FROM mb_pg.musicbrainz.artist_credit ac
    JOIN mb_pg.musicbrainz.artist_credit_name acn ON ac.id = acn.artist_credit
    JOIN mb_pg.musicbrainz.release_group rg ON rg.artist_credit = ac.id
    WHERE rg.type != 1
    ORDER BY ac.id
)
TO '../data/mb_artist_credit.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

In [ ]:
#import album artists
duck_con.execute(f"""
COPY (
    WITH artist_ranked AS (
        SELECT
            rg.id AS album_id,
            rg.name AS album_name,
            acn.artist AS artist_id,
            acn.name AS artist_name,
            CASE WHEN acn.artist = 1 OR acn.artist_credit = 1 THEN 1 ELSE 0 END AS is_va,
            ROW_NUMBER() OVER (
                PARTITION BY rg.id
                ORDER BY
                    CASE WHEN acn.artist != 1 AND acn.artist_credit != 1 THEN 0 ELSE 1 END ASC,
                    acn.position ASC
            ) AS rn
        FROM mb_pg.musicbrainz.release_group rg
        JOIN mb_pg.musicbrainz.artist_credit_name acn ON rg.artist_credit = acn.artist_credit
        WHERE rg.type = 1
    )
    SELECT DISTINCT
        album_id,
        album_name,
        CASE WHEN is_va = 0 THEN artist_id END AS artist_id,
        CASE WHEN is_va = 0 THEN artist_name END AS artist_name
    FROM artist_ranked
    WHERE rn = 1
    ORDER BY album_id
)
TO '../data/mb_album_artists.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")